# USIR v2.0 Experiment B: Table 2 Generation Control Pipeline

**Quantitative Verification of Autoregressive Long-Form Generation Control and Deadzone Suppression**  

This notebook autonomously executes the definitive Table 2 benchmark comparing:
1. `Standard Nucleus` ($p=0.9, T=0.8$)
2. `Min-p + Repetition Penalty` ($p_{min}=0.05, \text{rep}=1.15$)
3. `Reinforced Mode Regulation (RMR)` (KV-cache low-rank attenuation)
4. `Unconstrained BAOAB` (Constant friction without state-dependent gating)
5. **`USIR v2.0 Adaptive-BAOAB` (Proposed: $\Gamma_t(H_{syntax}) + $ Jacobian logit projector)**

**Metrics Evaluated (50 samples $\times$ 5 methods = 250 sequences, 3,000 tokens each)**:
- Correlation Dimension $D_2$ (Grassberger-Procaccia on hidden state trajectories)
- $1/f$ Spectral Discrepancy $|\beta - 1.0|$
- MAUVE Score against Aozora Bunko human literature corpus
- Syntactic Bracket Closure Rate (%)
- Reader Aesthetic Valence $V_t = \mathbb{E}[\max(0, -dF/dt)]$

In [ ]:
# Cell 1: Environment & GPU Auto-Detection
import subprocess
import sys
import torch

print("=" * 70)
print("USIR v2.0 Hardware Diagnostics and GPU Auto-Detection")
print("=" * 70)

gpu_name = "CPU"
vram_gb = 0.0
target_model = "Qwen/Qwen2.5-7B-Instruct"
quant_mode = "none"

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Detected GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    if vram_gb >= 35.0:  # A100 (40GB/80GB)
        target_model = "Qwen/Qwen2.5-14B-Instruct"
        quant_mode = "bf16"
    elif vram_gb >= 20.0:  # L4 (24GB)
        target_model = "Qwen/Qwen2.5-14B-Instruct"
        quant_mode = "8bit"
    elif vram_gb >= 12.0:  # T4 (16GB)
        target_model = "Qwen/Qwen2.5-7B-Instruct"
        quant_mode = "4bit"
    else:
        target_model = "Qwen/Qwen2.5-7B-Instruct"
        quant_mode = "4bit"
else:
    print("No CUDA GPU detected. Running in fast mock simulation mode.")

print(f"Selected Deployment Target: {target_model} [Quantization: {quant_mode}]")


In [ ]:
# Cell 2: Install Core Dependencies
!pip install -q transformers accelerate bitsandbytes scipy mauve-text janome matplotlib


In [ ]:
# Cell 3: Setup Working Directory and Python Path
import os
import sys

WORKSPACE_DIR = "/content/usir"
if not os.path.exists(WORKSPACE_DIR):
    if os.path.exists("python/colab/run_experiment_b_batch.py"):
        WORKSPACE_DIR = os.getcwd()
    elif os.path.exists("../../python/colab/run_experiment_b_batch.py"):
        WORKSPACE_DIR = os.path.abspath("../..")
        os.chdir(WORKSPACE_DIR)
    else:
        !git clone https://github.com/ughirose/usir.git /content/usir
        %cd /content/usir
        WORKSPACE_DIR = "/content/usir"

py_path = os.path.join(WORKSPACE_DIR, "python")
if py_path not in sys.path:
    sys.path.insert(0, py_path)

print(f"Active Workspace: {WORKSPACE_DIR}")


In [ ]:
# Cell 4: Execute Experiment B Batch Generation Pipeline
import subprocess

script_path = os.path.join(WORKSPACE_DIR, "python", "colab", "run_experiment_b_batch.py")
output_path = os.path.join(WORKSPACE_DIR, "artifacts", "table2_generation_control_results.json")

cmd = [
    sys.executable,
    script_path,
    "--model_name_or_path", target_model,
    "--samples_per_method", "50",
    "--num_tokens", "3000",
    "--output_path", output_path,
]

if quant_mode == "4bit":
    cmd.append("--load_in_4bit")
elif quant_mode == "8bit":
    cmd.append("--load_in_8bit")
elif gpu_name == "CPU":
    cmd.append("--mock_mode")

print("Running Pipeline:", " ".join(cmd))
res = subprocess.run(cmd, check=True)
print("Batch generation completed successfully!")


In [ ]:
# Cell 5: Visualize and Display Table 2
import json
from IPython.display import Markdown, display
from colab.merge_colab_results_to_paper import format_table2_markdown, format_table2_latex

results_file = os.path.join(WORKSPACE_DIR, "artifacts", "table2_generation_control_results.json")
with open(results_file, "r", encoding="utf-8") as f:
    results = json.load(f)

summary = results["table2_summary"]
display(Markdown("### Table 2: Generation Control Benchmark Results"))
display(Markdown(format_table2_markdown(summary)))

print("\n" + "=" * 70)
print("LaTeX Formatted Table 2 for Submission")
print("=" * 70)
print(format_table2_latex(summary))


In [ ]:
# Cell 6: Merge Results into Paper and Save to Drive
import shutil
from colab.merge_colab_results_to_paper import merge_results_to_paper

paper_file = os.path.join(WORKSPACE_DIR, "usir_v2_camera_ready_paper.md")
results_file = os.path.join(WORKSPACE_DIR, "artifacts", "table2_generation_control_results.json")
merge_results_to_paper(results_path=results_file, paper_path=paper_file)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = "/content/drive/MyDrive/usir_artifacts"
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy(results_file, os.path.join(drive_dir, "table2_generation_control_results.json"))
    shutil.copy(paper_file, os.path.join(drive_dir, "usir_v2_camera_ready_paper.md"))
    print(f"Exported artifacts to Google Drive: {drive_dir}")
except Exception as e:
    print(f"Local run or Drive not mounted: {e}. Artifacts preserved in workspace.")
